# Perhitungan Sentralitas

## Import Library

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

## Load Data

In [2]:
# pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/processed/edges.csv', header=None)
df.columns = ['Guru', 'Murid', 'Weight', 'Inverse_Weight']
df = df.iloc[1:].reset_index(drop=True)

display(df)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8148\3877324507.py:3: DtypeWarning: Columns (0: 2, 1: 3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/edges.csv', header=None)


,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1,1.0
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.0
2,جابر بن زيد,ابو عبيدة,522,0.0019157088122605363
3,عائشة,جابر بن زيد,68,0.014705882352941176
4,ابو هريرة,جابر بن زيد,76,0.013157894736842105
...,...,...,...,...
776810,ابي مسعود البدري,حذيفة بن اليمان,1,1.0
776811,سعيد بن العاص,ابي مسعود البدري,1,1.0
776812,ابو مسعود,سعيد بن العاص,1,1.0
776813,عبد الله,علي بن علقمة,1,1.0


In [3]:
print(df.dtypes)

Guru                 str
Murid                str
Weight            object
Inverse_Weight    object
dtype: object


In [4]:
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


### Bentuk graph

In [5]:
G = nx.from_pandas_edgelist(
    df,
    source='Murid',
    target='Guru',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

In [6]:
print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 177206
Jumlah edge : 776815


In [7]:
display(df)

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيدة,522,0.001916
3,عائشة,جابر بن زيد,68,0.014706
4,ابو هريرة,جابر بن زيد,76,0.013158
...,...,...,...,...
776810,ابي مسعود البدري,حذيفة بن اليمان,1,1.000000
776811,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776812,ابو مسعود,سعيد بن العاص,1,1.000000
776813,عبد الله,علي بن علقمة,1,1.000000


In [8]:
# Hapus self-loop
G.remove_edges_from(nx.selfloop_edges(G))

In [9]:
print("Jumlah node setelah hapus self-loop:", G.number_of_nodes())
print("Jumlah edge setelah hapus self-loop:", G.number_of_edges())

Jumlah node setelah hapus self-loop: 177206
Jumlah edge setelah hapus self-loop: 776815


## Menghitung Sentralitas

### Menghitung In Degree, Out Degree, Degree, dan Degree Centrality

In [10]:
# Degree (node atau total hubungan)
degree_dict = dict(G.degree())


# In-Degree (peran perawi sebagai Guru)
# In-Degree = jumlah murid
in_degree_dict = dict(G.in_degree())


# Out-Degree (peran perawi sebagai Murid)
# Out-Degree = jumlah guru
out_degree_dict = dict(G.out_degree())


# Degree Centrality (tingkat keterhubungan node dalam jaringan.)
degree_centrality_dict = nx.degree_centrality(G)

# Gabungkan ke DataFrame

centrality_df = pd.DataFrame({
    'Perawi': list(G.nodes()),

    'In_Degree_Jumlah_Murid': [
        in_degree_dict[node]
        for node in G.nodes()
    ],

    'Out_Degree_Jumlah_Guru': [
        out_degree_dict[node]
        for node in G.nodes()
    ],

    'Degree_Total_Hubungan': [
        degree_dict[node]
        for node in G.nodes()
    ],

    'Degree_Centrality': [
        degree_centrality_dict[node]
        for node in G.nodes()
    ]
})

# Urutkan berdasarkan Degree Centrality
centrality_df = centrality_df.sort_values(
    by='Degree_Centrality',
    ascending=False
)


# Reset index
centrality_df = centrality_df.reset_index(drop=True)


# Tampilkan hasil
display(centrality_df.head(20))

,Perawi,In_Degree_Jumlah_Murid,Out_Degree_Jumlah_Guru,Degree_Total_Hubungan,Degree_Centrality
0,ابو هريرة,3530,1291,4821,0.027206
1,سفيان,2294,2391,4685,0.026438
2,شعبة,2121,2198,4319,0.024373
3,ابن عباس,2662,946,3608,0.020361
4,عائشة,2334,733,3067,0.017308
5,الاعمش,1748,1306,3054,0.017234
6,ابن عمر,2246,796,3042,0.017167
7,ابو عبد الله الحافظ,786,2247,3033,0.017116
8,الزهري,1554,1466,3020,0.017042
9,عبد الله,1491,1364,2855,0.016111


### Menghitung Eigenvector Centrality (Kualitas koneksi)

In [11]:
# === Eigenvector Centrality (Power Iteration)
try:
    eigenvector_centrality = nx.eigenvector_centrality(
        G,
        max_iter=5000,
        tol=1e-05,
        weight='Weight'
    )
    print("✅ Eigenvector Centrality selesai.")
except nx.PowerIterationFailedConvergence as e:
    eigenvector_centrality = {}
    print(f"Proses Eigenvector Centrality gagal konvergen: {e}")

# Tetap memakai nama lama agar sel berikutnya masih kompatibel
eigenvector_dict = eigenvector_centrality

# Ubah hasil Power Iteration menjadi DataFrame
eigenvector_df = pd.DataFrame({
    'Perawi': list(eigenvector_centrality.keys()),
    'Eigenvector_Centrality': list(eigenvector_centrality.values())
})

if not eigenvector_df.empty:
    eigenvector_df = eigenvector_df.sort_values(
        by='Eigenvector_Centrality',
        ascending=False
    ).reset_index(drop=True)

display(eigenvector_df.head(20))


# === Eigenvector Centrality (NumPy version)
try:
    eigenvector_numpy_centrality = nx.eigenvector_centrality_numpy(
        G,
        weight='Weight'
    )
    print("✅ Eigenvector Numpy Centrality selesai.")
except Exception as e:
    eigenvector_numpy_centrality = {}
    print(f"Proses Eigenvector Numpy Centrality gagal: {e}")

# Ubah hasil NumPy menjadi DataFrame
eigenvector_numpy_df = pd.DataFrame({
    'Perawi': list(eigenvector_numpy_centrality.keys()),
    'Eigenvector_Numpy_Centrality': list(eigenvector_numpy_centrality.values())
})

if not eigenvector_numpy_df.empty:
    eigenvector_numpy_df = eigenvector_numpy_df.sort_values(
        by='Eigenvector_Numpy_Centrality',
        ascending=False
    ).reset_index(drop=True)

display(eigenvector_numpy_df.head(20))

,Perawi,Eigenvector_Centrality
0,ابو هريرة,0.517532
1,عائشة,0.516808
2,ابن عمر,0.342338
3,ابن عباس,0.266749
4,انس,0.174555
5,عمر,0.154329
6,انس بن مالك,0.131959
7,سعيد بن المسيب,0.124428
8,عروة,0.119143
9,الزهري,0.117185


In [12]:
# mengecek jumlah node setelah semua proses
print(G.number_of_nodes())

177206


In [13]:
# mengecek data  yang digunakan untuk menghitung eigenvector centrality
print(len(eigenvector_dict))

177206


### Menghitung data dengan data teratas untuk Closeness dan Between 

In [14]:
# # Ambil 40000 data teratas
# df_40000 = df.head(40000)

# # Buat graph dari kolom Guru dan Murid
# G_sample = nx.from_pandas_edgelist(
#     df_40000,
#     source='Guru',
#     target='Murid',
#     edge_attr=['Weight', 'Inverse_Weight'],
#     create_using=nx.DiGraph()
# )

# # Tampilkan jumlah node dan edge
# print("Jumlah Node :", G_sample.number_of_nodes())
# print("Jumlah Edge :", G_sample.number_of_edges())

In [15]:
# display(df_40000)

### Menghitung Closeness Centrality

In [27]:
import networkx as nx
import pandas as pd
from tqdm import tqdm

# ======================================
# Ambil 3000 data teratas
# ======================================

df_sample = df.head(20000).copy()

# ======================================
# Pastikan Inverse_Weight numerik
# ======================================

df_sample['Inverse_Weight'] = pd.to_numeric(
    df_sample['Inverse_Weight']
)

# ======================================
# Hindari nilai terlalu kecil
# ======================================

df_sample['Inverse_Weight'] = (
    df_sample['Inverse_Weight']
    .clip(lower=0.01)
)

# ======================================
# Build graph Guru -> Murid
# ======================================

G_sample = nx.from_pandas_edgelist(
    df_sample,
    source='Guru',
    target='Murid',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

print("Jumlah Node :", G_sample.number_of_nodes())
print("Jumlah Edge :", G_sample.number_of_edges())

# ======================================
# Hitung Closeness Centrality
# ======================================

raw_closeness = {}

for node in tqdm(
    G_sample.nodes(),
    desc="Menghitung Closeness Centrality"
):

    raw_closeness[node] = nx.closeness_centrality(
        G_sample,
        u=node,
        distance='Inverse_Weight'
    )

print("✅ Closeness Centrality selesai.")

# ======================================
# Normalisasi Min-Max (0 - 1)
# ======================================

min_val = min(raw_closeness.values())
max_val = max(raw_closeness.values())

normalized_closeness = {

    node: (
        (val - min_val) / (max_val - min_val)
        if max_val > min_val else 1.0
    )

    for node, val in raw_closeness.items()
}

print("✅ Normalisasi Closeness selesai.")

# ======================================
# Ubah hasil menjadi DataFrame
# ======================================

closeness_df = pd.DataFrame({
    'Perawi': list(normalized_closeness.keys()),
    'Closeness_Centrality': list(normalized_closeness.values())
})

# ======================================
# Urutkan dari terbesar
# ======================================

closeness_df = closeness_df.sort_values(
    by='Closeness_Centrality',
    ascending=False
).reset_index(drop=True)

# ======================================
# Tampilkan 20 teratas
# ======================================

display(closeness_df.head(20))

Jumlah Node : 7217
Jumlah Edge : 20000


Menghitung Closeness Centrality: 100%|██████████| 7217/7217 [15:15<00:00,  7.88it/s] 

✅ Closeness Centrality selesai.
✅ Normalisasi Closeness selesai.


,Perawi,Closeness_Centrality
0,سفيان,1.000000
1,شعبة,0.995665
2,مالك,0.995534
3,يحيي بن سعيد,0.993746
4,يونس,0.991054
5,كيع,0.990940
6,مالك بن انس,0.990252
7,عبد الرحمن بن مهدي,0.989091
8,محمد بن يوسف الفريابي,0.988958
9,عبد الله,0.988810


### Menghitung Between

In [31]:
import networkx as nx
import pandas as pd
from tqdm import tqdm
from datetime import datetime

# ======================================
# Ambil 3000 data teratas
# ======================================

df_sample = df.head(20000).copy()

# ======================================
# Pastikan Inverse_Weight numerik
# ======================================

df_sample['Inverse_Weight'] = pd.to_numeric(
    df_sample['Inverse_Weight']
)

# ======================================
# Hindari nilai terlalu kecil
# ======================================

df_sample['Inverse_Weight'] = (
    df_sample['Inverse_Weight']
    .clip(lower=0.01)
)

# ======================================
# Build graph Guru -> Murid
# ======================================

G = nx.from_pandas_edgelist(
    df_sample,
    source='Guru',
    target='Murid',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

print("✅ Graph berhasil dibuat")
print("Jumlah Node :", G.number_of_nodes())
print("Jumlah Edge :", G.number_of_edges())

# ======================================
# Timestamp mulai
# ======================================

start_time = datetime.now()

print(f"🚀 Betweenness Centrality started at: {start_time}")

# ======================================
# Fungsi Betweenness Centrality
# dengan progress bar
# ======================================

def betweenness_centrality_with_progress(
    G,
    normalized=True,
    weight=None
):

    nodes = list(G.nodes())

    bet = dict.fromkeys(nodes, 0.0)

    for s in tqdm(
        nodes,
        desc="Betweenness Centrality (Nodes)"
    ):

        # Hitung kontribusi betweenness
        contrib = nx.betweenness_centrality_subset(
            G,
            sources=[s],
            targets=nodes,
            normalized=normalized,
            weight=weight
        )

        # Gabungkan kontribusi
        for n, v in contrib.items():

            bet[n] += v

    return bet

# ======================================
# Hitung Betweenness Centrality
# ======================================

bet_node = betweenness_centrality_with_progress(
    G,
    normalized=True,
    weight='Inverse_Weight'
)

print("✅ Betweenness Centrality selesai.")

# ======================================
# Timestamp selesai
# ======================================

end_time = datetime.now()

print(f"🏁 Finished at: {end_time}")
print(f"⏱️ Duration: {end_time - start_time}")

# ======================================
# Ubah hasil menjadi DataFrame
# ======================================

betweenness_df = pd.DataFrame({
    'Perawi': list(bet_node.keys()),
    'Betweenness_Centrality': list(bet_node.values())
})

# ======================================
# Urutkan dari terbesar
# ======================================

betweenness_df = betweenness_df.sort_values(
    by='Betweenness_Centrality',
    ascending=False
).reset_index(drop=True)

# ======================================
# Tampilkan 20 teratas
# ======================================

display(betweenness_df.head(20))

✅ Graph berhasil dibuat
Jumlah Node : 7217
Jumlah Edge : 20000
🚀 Betweenness Centrality started at: 2026-05-19 00:32:25.183334


Betweenness Centrality (Nodes): 100%|██████████| 7217/7217 [06:07<00:00, 19.66it/s] 

✅ Betweenness Centrality selesai.
🏁 Finished at: 2026-05-19 00:38:32.335599
⏱️ Duration: 0:06:07.152265


,Perawi,Betweenness_Centrality
0,سفيان,0.450989
1,عبد الله,0.185856
2,ابو حنيفة,0.141320
3,ابو هريرة,0.131439
4,علي,0.124418
5,يحيي بن سعيد,0.109814
6,محمد بن ابراهيم,0.082643
7,ابن عباس,0.080035
8,عمر,0.076862
9,شعبة,0.071216
